# 03 - HAAQI Score
Compute HAAQI score for the baseline output 
Run `01_setup.ipynb` first (needs `clarity/` cloned and installed).

##### Needs about 30 GB of free disk and a long first 12.6 GB download

1. Download the valid-set data i.e. three files from Zenodo

In [ ]:
import hashlib, subprocess
from pathlib import Path

ZENODO = 'https://zenodo.org/api/records/14335552/files'
FILES = {
    'metadata.zip': '7ff49002ae62023cd547ff057cbba461',
    'EnsembleSet_Mix_1.zip': '5bdcd0154da69e7516aef2d3062a47f0',
    'CadenzaWoodwind.zip': 'cdc08be8482662c5023b5d4060d1b064',
}
dl = Path('zenodo_download').resolve()
dl.mkdir(exist_ok=True)

def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

for name, want in FILES.items():
    path = dl / name
    if not path.exists() or md5sum(path) != want:
        subprocess.run(['wget', '-q', '-c', '-O', str(path), f'{ZENODO}/{name}/content'], check=True)
    assert md5sum(path) == want, f'md5 mismatch for {name}'
    print('OK', name)

OK metadata.zip


2. Build the dataset. Clarity's script unzips the files and creates the `audio/` and `metadata/` folders

In [ ]:
root = Path('cadenza_data/cad2/task2').resolve()
clarity_dir = Path('clarity').resolve()

# Colab's torchtune has its own `recipes` package; PYTHONPATH makes Clarity's win.
!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/process_dataset/process_zenodo_download.py \
    path.zenodo_download_path={dl} \
    path.root={root}

In [ ]:
meta = root / 'metadata'
for name in ['scenes.valid.json', 'scene_listeners.valid.json', 'music.valid.json', 'listeners.valid.json']:
    assert (meta / name).exists(), f'missing {name}'
print('valid audio folders:', sorted(p.name for p in (root / 'audio' / 'valid').iterdir())[:5], '...')

3. Pick a first scene in the valid set with its first listener

In [ ]:
import json

scenes = json.load(open(meta / 'scenes.valid.json'))
scene_listeners = json.load(open(meta / 'scene_listeners.valid.json'))

scene_id = next(iter(scene_listeners))
listener_id = scene_listeners[scene_id][0]
print('scene   :', scene_id, scenes[scene_id])
print('listener:', listener_id)

one = root / 'metadata_one_scene'
one.mkdir(exist_ok=True)
json.dump({scene_id: scenes[scene_id]}, open(one / 'scenes.json', 'w'))
json.dump({scene_id: [listener_id]}, open(one / 'scene_listeners.json', 'w'))

4. Run the baseline - Same as notebook 02 but on the valid set.

In [ ]:
exp = Path('exp_valid_one_scene').resolve()
overrides = f'path.root={root} path.scenes_file={one}/scenes.json path.scene_listeners_file={one}/scene_listeners.json path.exp_folder={exp}'

!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/enhance.py {overrides}

5. Compute HAAQI - Rebuilds the reference from the stems and then runs it through the listener's hearing-aid model, then compares it to our enhanced signal for each ear.

In [ ]:
!PYTHONPATH={clarity_dir} python clarity/recipes/cad2/task2/baseline/evaluate.py {overrides}

6. Read the score - HAAQI  between 0 and 1, higher the better.

In [ ]:
import csv

with open(exp / 'scores.csv') as f:
    rows = list(csv.DictReader(f))
for r in rows:
    print(f"scene {r['scene']} | listener {r['listener']} | song {r['song']}")
    print(f"left: {float(r['left_haaqi']):.4f}")
    print(f"right: {float(r['right_haaqi']):.4f}")
    print(f"avg: {float(r['avg_haaqi']):.4f}")

## DONE
If a score printed, HAAQI is computed for the baseline output.